# Nutrient Composition Measurements of Cooked Foods Commonly Consumed by African-born Immigrants in the United States Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading, exploring, and processing the FAIR^2 nutrient composition dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.1gk3-9cbm/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.1gk3-9cbm/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset name:", metadata.name)
print("Description:", metadata.description)
print("Version:", metadata.version)
print("Published date:", metadata.datePublished)

## 2. Data Overview
Review available record sets (`cr:RecordSet`), fields (`cr:Field`), and columns (`cr:column`) using their `@id` references.

We will list all record sets found in the dataset, and for each, enumerate their fields and columns, referencing everything by `@id`.

In [ ]:
# Get all record sets' @id
record_sets_metadata = dataset.metadata.recordSet
if not record_sets_metadata:
    print("No record sets found in dataset metadata.")
else:
    for rs in record_sets_metadata:
        print(f"RecordSet @id: {rs['@id']}")
        fields = rs.get('field', [])
        if fields:
            print("  Fields:")
            for field in fields:
                print(f"    Field @id: {field['@id']}, name: {field.get('name', '')}, dataType: {field.get('dataType', '')}")
                columns = field.get('column', [])
                if columns:
                    print("      Columns:")
                    for col in columns:
                        print(f"        Column @id: {col['@id']}, name: {col.get('name','')}")
        else:
            print("  No fields listed in this RecordSet.")


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

First, let's store all record set IDs for extraction.

In [ ]:
# Gather all record set @id values
record_sets = []
if dataset.metadata.recordSet:
    for rs in dataset.metadata.recordSet:
        record_sets.append(rs['@id'])
else:
    print("No record sets found in the metadata.")

dataframes = {}
for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)

# Display columns of the first record set, and show sample
if record_sets:
    first_rs = record_sets[0]
    print(f"Columns for RecordSet @id: {first_rs}")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will:
- Select a numeric field by its `@id`
- Filter rows for values above a threshold
- Normalize the numeric field
- Optionally group by a categorical field (e.g., food name) using its `@id`

In [ ]:
# Choose a record set to analyze (first one for demonstration)
record_set_id = record_sets[0] if record_sets else None
df = dataframes[record_set_id]

# List available columns for inspection, referencing by @id
print("Available columns in DataFrame:")
print(df.columns.tolist())

# For illustration, suppose the numeric field is 'cr:Energy_kcal_100g' and the group field is 'cr:Food_Name'
# If these columns are present, use them; otherwise, select a numeric field and a food/categorical field

# Try to detect some numeric and group (categorical) fields automatically
numeric_candidates = [col for col in df.columns if col.endswith("kcal") or col.endswith("g") or col.lower().startswith("cr:") or "energy" in col.lower()]
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    numeric_field_id = df.select_dtypes(include='number').columns[0] if len(df.select_dtypes(include='number').columns) > 0 else df.columns[0]

group_field_candidates = [col for col in df.columns if "name" in col.lower() or "food" in col.lower()]
if group_field_candidates:
    group_field_id = group_field_candidates[0]
else:
    group_field_id = df.columns[0]

print(f"Selected numeric field (@id): {numeric_field_id}")
print(f"Selected group field (@id): {group_field_id}")

# Set threshold, filter, normalize, and group
threshold = 10
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id} (showing mean of numeric columns):")
        display(grouped_df.head())
else:
    print(f"Numeric field {numeric_field_id} not found in columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll show a histogram of the selected numeric field and a boxplot grouped by the chosen categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Boxplot by group field
if group_field_id in df.columns:
    plt.figure(figsize=(12,6))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.xticks(rotation=90)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we used `mlcroissant` to load and explore the FAIR^2 nutrient composition dataset for cooked foods consumed by African-born immigrants in the United States. We:
- Inspected dataset metadata
- Listed available record sets, fields, and columns referenced by `@id`
- Extracted records into DataFrames
- Performed filtering, normalization, and grouping on numeric fields
- Visualized distributions and group relationships

This approach offers a reproducible, standards-based workflow for FAIR dataset exploration and paves the way for downstream nutritional research, dietary modeling, and policy analysis. Explore additional fields and analyses by referencing their `@id` in notebook cells.